In [1]:
print("Python is running inside VS Code")

Python is running inside VS Code


In [2]:
# Week 2 – Gradient Descent & Backpropagation
# CST 618 – Training ML & LLMs

import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report



In [3]:
# Load dataset
data = load_wine()

df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [4]:
# Step 3: Split dataset and scale data
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Train shape: (124, 13)
Test shape: (54, 13)


In [5]:
def report(model, label):
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    print(f"\n===== {label} =====")
    print(classification_report(y_test, preds, digits=4))
    return preds

In [6]:
# Step 4: Stochastic Gradient Descent (SGD)
sgd_model = MLPClassifier(
    hidden_layer_sizes=(50,),
    solver="sgd",
    batch_size=1,
    learning_rate_init=0.01,
    max_iter=1000,
    random_state=42
)

pred_sgd = report(sgd_model, "SGD (batch_size=1)")


===== SGD (batch_size=1) =====
              precision    recall  f1-score   support

           0     0.9474    1.0000    0.9730        18
           1     0.9524    0.9524    0.9524        21
           2     1.0000    0.9333    0.9655        15

    accuracy                         0.9630        54
   macro avg     0.9666    0.9619    0.9636        54
weighted avg     0.9639    0.9630    0.9629        54



In [7]:
# Step 5: Batch Gradient Descent (full batch updates)
batch_model = MLPClassifier(
    hidden_layer_sizes=(50,),
    solver="sgd",
    batch_size=X_train_scaled.shape[0],
    learning_rate_init=0.01,
    max_iter=1000,
    random_state=42
)

pred_batch = report(batch_model, "Batch GD (full batch)")


===== Batch GD (full batch) =====
              precision    recall  f1-score   support

           0     0.9474    1.0000    0.9730        18
           1     0.9524    0.9524    0.9524        21
           2     1.0000    0.9333    0.9655        15

    accuracy                         0.9630        54
   macro avg     0.9666    0.9619    0.9636        54
weighted avg     0.9639    0.9630    0.9629        54



In [8]:
# Step 6: Early stopping + activation comparisons

relu_es = MLPClassifier(
    hidden_layer_sizes=(50,),
    solver="sgd",
    batch_size=32,
    activation="relu",
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    learning_rate_init=0.01,
    max_iter=2000,
    random_state=42
)

tanh_es = MLPClassifier(
    hidden_layer_sizes=(50,),
    solver="sgd",
    batch_size=32,
    activation="tanh",
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    learning_rate_init=0.01,
    max_iter=2000,
    random_state=42
)

pred_relu = report(relu_es, "Early Stopping + ReLU")
pred_tanh = report(tanh_es, "Early Stopping + tanh")


===== Early Stopping + ReLU =====
              precision    recall  f1-score   support

           0     0.9444    0.9444    0.9444        18
           1     0.9524    0.9524    0.9524        21
           2     1.0000    1.0000    1.0000        15

    accuracy                         0.9630        54
   macro avg     0.9656    0.9656    0.9656        54
weighted avg     0.9630    0.9630    0.9630        54


===== Early Stopping + tanh =====
              precision    recall  f1-score   support

           0     0.9474    1.0000    0.9730        18
           1     0.9500    0.9048    0.9268        21
           2     0.9333    0.9333    0.9333        15

    accuracy                         0.9444        54
   macro avg     0.9436    0.9460    0.9444        54
weighted avg     0.9445    0.9444    0.9440        54



In [9]:
# Step 7: Grid Search for hyperparameter tuning

base = MLPClassifier(
    hidden_layer_sizes=(50,),
    solver="sgd",
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    max_iter=2000,
    random_state=42
)

param_grid = {
    "learning_rate_init": [0.001, 0.005, 0.01, 0.05],
    "batch_size": [16, 32, 64, 128],
    "activation": ["relu", "tanh"]
}

grid = GridSearchCV(base, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print("Best Grid Params:", grid.best_params_)
print("Best Grid CV Score:", grid.best_score_)

/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_mul

Best Grid Params: {'activation': 'relu', 'batch_size': 16, 'learning_rate_init': 0.05}
Best Grid CV Score: 0.9756666666666666


/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(


In [10]:
# Step 8: Random Search for hyperparameter tuning

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

param_dist = {
    "learning_rate_init": uniform(0.001, 0.1),
    "batch_size": [16, 32, 64, 128],
    "activation": ["relu", "tanh"]
}

random_search = RandomizedSearchCV(
    base,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_scaled, y_train)

print("Best Random Params:", random_search.best_params_)
print("Best Random CV Score:", random_search.best_score_)

/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_mul

Best Random Params: {'activation': 'relu', 'batch_size': 128, 'learning_rate_init': np.float64(0.09607143064099162)}
Best Random CV Score: 0.9756666666666666


/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:695: UserWarning: Got `batch_size` less than 1 or larger than sample size. It is going to be clipped
  warnings.warn(
/Users/boatismbair/Desktop/New Project Directory CST 618 100/CST618 Wine ML Week2/ml-env/lib/python3.14/site-packages/sklearn/neural_network/_mul

In [11]:
# Step 9: K-Fold Cross-Validation for model evaluation
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

# Choose best model based on CV score (grid vs random)
best_model = grid.best_estimator_ if grid.best_score_ >= random_search.best_score_ else random_search.best_estimator_
print("Chosen best model:", best_model)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=kf, scoring="accuracy")

print("K-Fold CV Scores:", cv_scores)
print("Mean CV Accuracy:", np.mean(cv_scores))
print("Std Dev:", np.std(cv_scores))

Chosen best model: MLPClassifier(batch_size=16, early_stopping=True, hidden_layer_sizes=(50,),
              learning_rate_init=0.05, max_iter=2000, n_iter_no_change=20,
              random_state=42, solver='sgd', validation_fraction=0.15)
K-Fold CV Scores: [1.    1.    0.96  1.    0.875]
Mean CV Accuracy: 0.967
Std Dev: 0.048538644398046386


In [12]:
# Step 10: Final model evaluation on test data
from sklearn.metrics import accuracy_score, classification_report

best_model.fit(X_train_scaled, y_train)
final_preds = best_model.predict(X_test_scaled)

print("Final Test Accuracy:", accuracy_score(y_test, final_preds))
print(classification_report(y_test, final_preds, digits=4))

Final Test Accuracy: 0.9629629629629629
              precision    recall  f1-score   support

           0     0.9444    0.9444    0.9444        18
           1     0.9524    0.9524    0.9524        21
           2     1.0000    1.0000    1.0000        15

    accuracy                         0.9630        54
   macro avg     0.9656    0.9656    0.9656        54
weighted avg     0.9630    0.9630    0.9630        54

